cromadb is a vector database


In [1]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & GOOGLE GEMINI API AUTHENTICATION
# Run this cell first. Installs the official Google GenAI SDK.
# ==============================================================================
!pip install -q -U google-genai tabulate

import os
import getpass
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Modern Google GenAI SDK
from google import genai
from google.genai import types

# Aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 4)

# ------------------------------------------------------------------------------
# Secure Gemini API Key Ingestion
# ------------------------------------------------------------------------------
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("🔑 Enter your Google Gemini API Key: ")
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY

# Initialize Client
client = genai.Client(api_key=GEMINI_API_KEY)
print("✅ Google Gemini Agentic Client initialized successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 794.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
✅ Google Gemini Agentic Client initialized successfully!


In [ ]:
# ==============================================================================
# SECTION 1: FROM CHATBOT TO AGENT (RESPONDING VS. ACTING)
# ==============================================================================
"""
THE EVOLUTION OF LLM APPLICATIONS:

1. Chatbot (Responding):
   - Confined to its training data context window.
   - Output: Text, code, or formatting.
   - Weakness: Cannot interact with the outside world, hallucinate recent facts, cannot take action.

2. Agent (Acting via Tool Calling):
   - Has a "brain" (LLM) connected to "hands" (Tools/Functions).  # astra and antigravoity
   - Can pause generation, request a function execution (e.g., run SQL, search the web, trigger an API),
     wait for the result, and synthesize the final answer.

--------------------------------------------------------------------------------
THE ReAct PATTERN (Reason + Act + Observe)
--------------------------------------------------------------------------------
Agents operate in a continuous cognitive loop developed by Princeton/Google researchers:
1. REASON (Thought): "The user wants to know Apple's current stock price and multiply it by 50."
2. ACT (Action): `get_stock_price(ticker="AAPL")`
3. OBSERVE (Observation): The function returns `150.25`.
4. REASON (Thought 2): "Now I need to multiply 150.25 by 50."
5. ACT (Action 2): `calculate(expression="150.25 * 50")`
6. OBSERVE (Observation 2): The function returns `7512.50`.
7. SYNTHESIZE: "If you buy 50 shares of AAPL, it will cost $7,512.50."
"""

'\nTHE EVOLUTION OF LLM APPLICATIONS:\n\n1. Chatbot (Responding):\n   - Confined to its training data context window.\n   - Output: Text, code, or formatting.\n   - Weakness: Cannot interact with the outside world, hallucinate recent facts, cannot take action.\n\n2. Agent (Acting via Tool Calling):\n   - Has a "brain" (LLM) connected to "hands" (Tools/Functions).\n   - Can pause generation, request a function execution (e.g., run SQL, search the web, trigger an API), \n     wait for the result, and synthesize the final answer.\n\n--------------------------------------------------------------------------------\nTHE ReAct PATTERN (Reason + Act + Observe)\n--------------------------------------------------------------------------------\nAgents operate in a continuous cognitive loop developed by Princeton/Google researchers:\n1. REASON (Thought): "The user wants to know Apple\'s current stock price and multiply it by 50."\n2. ACT (Action): `get_stock_price(ticker="AAPL")`\n3. OBSERVE (Obse

In [ ]:
# ==============================================================================
# SECTION 2: TOOL CALLING MECHANICS (DEFINING THE SCHEMA)
# ==============================================================================
"""
HOW THE MODEL DECIDES TO CALL A TOOL:
1. We pass a list of Python functions (or JSON schemas) in the API request configuration.
2. The LLM evaluates the user prompt against the descriptions of the provided tools.

# tools are the connectors here

3. If a tool is needed, the LLM stops returning standard text and instead returns a `FunctionCall`
   payload containing the function name and the extracted arguments (e.g., `{"ticker": "AAPL"}`).
4. The execution environment (your Python script) intercepts this, runs the actual code,
   and returns a `FunctionResponse` back to the LLM.
5. The LLM reads the result and resumes answering.

NOTE: Python Type Hints and Docstrings are MANDATORY! Gemini uses them to build the tool schema automatically.
"""

# Tool 1: Live Database Lookup Simulation
def get_customer_balance(customer_id: str) -> float:
    """Looks up a customer's current account balance in the database.

    Args:
        customer_id: The unique customer identifier (e.g., 'CUST-100').
    """
    print(f"\n⚙️  [SYSTEM LOG] Executing DB Query for {customer_id}...")
    # Simulated database
    db = {"CUST-100": 2500.50, "CUST-200": -450.00, "CUST-300": 0.0}
    return db.get(customer_id.upper(), 0.0)   #0.0 show the format bas 0.0

# Tool 2: Live Calculator
def safe_calculator(expression: str) -> float:
    """Evaluates a mathematical expression and returns the numeric result.

    Args:
        expression: A valid mathematical string (e.g., '2500.50 * 0.05').
    """
    print(f"⚙️  [SYSTEM LOG] Calculating: {expression}...")
    try:
        # In production, NEVER use raw eval(). Use safe parsers (e.g., numexpr or AST).
        return float(eval(expression))
    except Exception as e:
        return 0.0

print("✅ Tools defined: get_customer_balance, safe_calculator")

✅ Tools defined: get_customer_balance, safe_calculator


In [ ]:
# ==============================================================================
# SECTION 3: LIVE DEMO — MULTI-STEP TASK DECOMPOSITION & EXECUTION TRACE
# ==============================================================================
# We initialize an automated chat session. Gemini SDK handles the ReAct
# loop (calling the function and returning the result) automatically!

print("=== 🤖 INITIATING AGENTIC CHAT SESSION ===")

agent_chat = client.chats.create(
    model="gemini-3.6-flash",
    config=types.GenerateContentConfig(
        tools=[get_customer_balance, safe_calculator],   # veyr imp to f=define these while making an agent
        temperature=0.0, # Deterministic temperature for tool calling!
    )
)

user_prompt = "What is the balance of CUST-100? If I apply a 7.5% interest rate to it, what will the new total be?"
print(f"👤 User: {user_prompt}\n")

# Send the message. The SDK will automatically pause, run the Python functions, and send the results back!
response = agent_chat.send_message(user_prompt)

print(f"\n🤖 Final Agent Output:\n{response.text}\n")

# ------------------------------------------------------------------------------
# TRACING THE MODEL's REASONING STEPS
# ------------------------------------------------------------------------------
print("=== 🔍 SYSTEM AUDIT: REASONING & EXECUTION TRACE ===")

# We iterate through the raw conversation history to see what the model did behind the scenes.
for i, message in enumerate(agent_chat.get_history()):
    role_emoji = "👤 User" if message.role == "user" else "🤖 Model"
    print(f"Step {i+1} [{role_emoji}]:")

    for part in message.parts:
        # Check if the part is a standard text response
        if part.text:
            print(f"   [Text] -> {part.text.strip()}")

        # Check if the part is a request to call a tool
        elif part.function_call:
            print(f"   [Action Requested] -> {part.function_call.name}({part.function_call.args})")

        # Check if the part is the execution result returned to the model
        elif part.function_response:
            print(f"   [Observation Received] -> {part.function_response.name}: {part.function_response.response}")
    print("-" * 50)

=== 🤖 INITIATING AGENTIC CHAT SESSION ===
👤 User: What is the balance of CUST-100? If I apply a 7.5% interest rate to it, what will the new total be?


⚙️  [SYSTEM LOG] Executing DB Query for CUST-100...
⚙️  [SYSTEM LOG] Calculating: 2500.5 * 1.075...

🤖 Final Agent Output:
The current balance for **CUST-100** is **$2,500.50**.

Applying a 7.5% interest rate to this balance gives a new total of **$2,688.04** (exact: $2,688.0375).

=== 🔍 SYSTEM AUDIT: REASONING & EXECUTION TRACE ===
Step 1 [👤 User]:
   [Text] -> What is the balance of CUST-100? If I apply a 7.5% interest rate to it, what will the new total be?
--------------------------------------------------
Step 2 [🤖 Model]:
   [Action Requested] -> get_customer_balance({'customer_id': 'CUST-100'})
--------------------------------------------------
Step 3 [👤 User]:
   [Observation Received] -> get_customer_balance: {'result': 2500.5}
--------------------------------------------------
Step 4 [🤖 Model]:
   [Action Requested] -> safe_ca

In [ ]:
# ==============================================================================
# SECTION 4: AGENTIC ORCHESTRATION, MEMORY, & GUARDRAILS
# ==============================================================================
"""
LANGCHAIN CORE COMPONENTS:
While you can build agents manually (as seen above), frameworks like LangChain abstract the boilerplate.
- Prompts: Templates for standardizing inputs.
- Chains: Hardcoded sequential pipes (Prompt -> Model -> Output Parser).
- Retrievers: Connections to Vector DBs for RAG.
- Agents: Dynamic chains where the LLM decides the routing flow based on tools.
- Trade-off: Frameworks speed up development but introduce heavy abstractions that can be hard to debug compared to raw API calls.

LANGGRAPH & MULTI-AGENT SYSTEMS:
- A state-machine approach to agents.
- Defines workflows as Graphs (Nodes = Agents/Tools, Edges = Routing Logic).
- Enables cycles (Agent A asks Agent B for code, Agent B writes it, Agent A tests it and sends it back if it fails).
- Introduces "Supervisor Agents" that delegate tasks to "Worker Agents".

GUARDRAILS & RUNAWAY LOOP PREVENTION:
1. Infinite Loop Prevention: An agent might call a failing tool over and over. Always implement `max_iterations` (e.g., force stop after 5 tool calls).
2. Cost Constraints: Tool loops consume massive tokens. Set budget caps.
3. Input/Output Filtering: Validate function arguments before execution to prevent Prompt Injection (e.g., a user prompt instructing the DB tool to run `DROP TABLE`).
4. Scope Limiting: Only give agents read-only tools unless operating in a highly trusted sandboxed environment.
"""

In [3]:
# ==============================================================================
# SECTION 5: STUDENT GRADED LAB — BUILD YOUR OWN TOOL
# ==============================================================================
"""
🎓 STUDENT LAB INSTRUCTIONS:
Build a working agent capable of fetching real-time/mock data using an external function.

MANDATORY TASKS:
1. Define a custom Python function with proper Type Hints and Docstrings.
   (e.g., `check_flight_status(flight_number: str)` or `get_inventory(product_id: int)`)
2. Create a Gemini Chat session injecting your custom tool into the `GenerateContentConfig`.
3. Send a user message that forces the model to use your tool to answer the question.
4. Extract and print the final text response.
"""

# ------------------------------------------------------------------------------
# STUDENT WORKSPACE (WRITE YOUR SOLUTION BELOW)
# ------------------------------------------------------------------------------

# 1. Define your custom tool (Function + Docstring)
# TODO: Write a function that simulates checking a flight status, weather, or inventory.
# Tool 1: Live Database Lookup Simulation


def check_flight_status(flight_number: str) -> float:


    print(f"\n⚙️  [SYSTEM LOG] Executing DB Query for {flight_number}...")
    # Simulated database
    db = {"flight1delay": 15.45, "DL404": 25.35, "flight3delay": 0.0}
    return db.get(flight_number.upper(), 0.0)

# Tool 2: Live Calculator
def get_inventory(product_id: int) -> float:

    print(f"⚙️  [SYSTEM LOG] Calculating: {product_id}...")
    try:

        return float(eval(product_id))
    except Exception as e:
        return 0.0

# 2. Initialize Agent Chat Session
# TODO: Create client.chats.create() and pass your function inside the tools list.
agent_chat = client.chats.create(
    model="gemini-3.6-flash",
    config=types.GenerateContentConfig(
        tools=[check_flight_status, get_inventory],
        temperature=0.0, # Deterministic temperature for tool calling!
    )
)


# 3. Trigger the Agent
# TODO: Send a message asking for the status of flight 'DL404'.
user_prompt = "What is the status of flight 'DL404'? "
print(f"👤 User: {user_prompt}\n")
response = agent_chat.send_message(user_prompt)



# 4. Output the result

for i, message in enumerate(agent_chat.get_history()):
    role_emoji = "👤 User" if message.role == "user" else "🤖 Model"
    print(f"Step {i+1} [{role_emoji}]:")

    for part in message.parts:
        # Check if the part is a standard text response
        if part.text:
            print(f"   [Text] -> {part.text.strip()}")

        # Check if the part is a request to call a tool
        elif part.function_call:
            print(f"   [Action Requested] -> {part.function_call.name}({part.function_call.args})")

        # Check if the part is the execution result returned to the model
        elif part.function_response:
            print(f"   [Observation Received] -> {part.function_response.name}: {part.function_response.response}")
    print("-" * 50)


    #final o/p
    print(f"\n🤖 Final Agent Output:\n{response.text}\n")


👤 User: What is the status of flight 'DL404'? 


⚙️  [SYSTEM LOG] Executing DB Query for DL404...
Step 1 [👤 User]:
   [Text] -> What is the status of flight 'DL404'?
--------------------------------------------------

🤖 Final Agent Output:
The status check for flight **DL404** returned a value of **25.35**.

Step 2 [🤖 Model]:
   [Action Requested] -> check_flight_status({'flight_number': 'DL404'})
--------------------------------------------------

🤖 Final Agent Output:
The status check for flight **DL404** returned a value of **25.35**.

Step 3 [👤 User]:
   [Observation Received] -> check_flight_status: {'result': 25.35}
--------------------------------------------------

🤖 Final Agent Output:
The status check for flight **DL404** returned a value of **25.35**.

Step 4 [🤖 Model]:
   [Text] -> The status check for flight **DL404** returned a value of **25.35**.
--------------------------------------------------

🤖 Final Agent Output:
The status check for flight **DL404** returned a val